In [ ]:

#EDAD
import pandas as pd

# Config

MIN_SCORE = 0.85


# Cargar dataset NER
df_ner = pd.read_csv(NER_DATASET, encoding='utf-8')
print(f"Entidades NER: {len(df_ner)}")

# Cargar dataset completo
df_data = pd.read_csv(RECURRENCE_DATASET, encoding='utf-8')
print(f"Consultas: {len(df_data)}")
print(f"Columnas: {list(df_data.columns)}")

print(f"\nFiltrando AGE con score >= {MIN_SCORE}...")
df_age = df_ner[(df_ner['label'] == 'AGE') & (df_ner['score'] >= MIN_SCORE)].copy()
print(f"Entidades AGE con score alto: {len(df_age)}")


df_merged = df_age.merge(
    df_data,
    left_on='patient_id',
    right_on='ID_PACIENTE',
    how='inner'
)

print(f"Entidades tras merge: {len(df_merged)}")

df_final = df_merged[df_merged['RECURRENCIA'] == 1].copy()

print(f"\nResultado final:")
print(f"  Entidades AGE (score >= {MIN_SCORE}) con recurrencia positiva: {len(df_final)}")
print(f"  Consultas unicas: {df_final['ID_PACIENTE'].nunique()}")

if len(df_final) > 0:
    print(f"\nEstadisticas score:")
    print(f"  Promedio: {df_final['score'].mean():.3f}")
    print(f"  Minimo: {df_final['score'].min():.3f}")
    print(f"  Maximo: {df_final['score'].max():.3f}")
    
    print("\nPrimeros 10 ejemplos:")
    cols_display = ['ID_PACIENTE', 'span', 'score', 'RECURRENCIA', 'FOLD']
    print(df_final[cols_display].head(10).to_string(index=False))
    
    # Guardar
    df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print(f"\nArchivo guardado: {OUTPUT_CSV}")
else:
    print("\nNo se encontraron resultados")


In [ ]:
import pandas as pd
df2 = pd.read_csv("age_entities_recurrence_positive.csv")
df2.head()

In [3]:
import pandas as pd
#config
df = pd.read_csv(INPUT_CSV, encoding='utf-8')
print(f"Total filas: {len(df)}")
df['id_paciente_36'] = df['ID_PACIENTE'].str[:36]

print(f"Pacientes unicos: {df['id_paciente_36'].nunique()}")
df_best = df.loc[df.groupby('id_paciente_36')['score'].idxmax()]

print(f"\nResultado:")
print(f"  Filas finales (1 por paciente): {len(df_best)}")
print(f"  Score promedio: {df_best['score'].mean():.3f}")
print(f"  Score minimo: {df_best['score'].min():.3f}")

cols_display = ['id_paciente_36', 'span', 'score', 'RECURRENCIA']
df_best.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nArchivo guardado: {OUTPUT_CSV}")


Total filas: 224
Pacientes unicos: 162

Resultado:
  Filas finales (1 por paciente): 162
  Score promedio: 0.986
  Score minimo: 0.864

Archivo guardado: age_best_per_patient.csv


In [4]:
import pandas as pd

# Config
AGE_DATASET = "age_best_per_patient.csv"  # Archivo con edades únicas por paciente
DATES_DATASET = "recurrence_data_cleaned.csv"  # Archivo con fechas separadas
OUTPUT_CSV = "patient_timeline_recurrence.csv"

df_age = pd.read_csv(AGE_DATASET, encoding='utf-8')
print(f"Pacientes con edad detectada: {len(df_age)}")
df_dates = pd.read_csv(DATES_DATASET, encoding='utf-8')
print(f"Total consultas: {len(df_dates)}")

if 'id_paciente_36' not in df_age.columns:
    df_age['id_paciente_36'] = df_age['ID_PACIENTE'].str[:36]
df_dates['fecha_dt'] = pd.to_datetime(df_dates['fecha'], format='%Y/%m/%d', errors='coerce')


results = []

for idx, row in df_age.iterrows():
    paciente_id = row['id_paciente_36']
    edad = row['span']
    
    consultas_paciente = df_dates[df_dates['id_paciente'] == paciente_id].copy()
    
    if len(consultas_paciente) == 0:
        print(f"  Advertencia: No se encontraron consultas para el {paciente_id}")
        continue
    
    consultas_paciente = consultas_paciente.sort_values('fecha_dt')
    primera_consulta = consultas_paciente.iloc[0]
    fecha_primera = primera_consulta['fecha']
    
    consultas_recurrencia = consultas_paciente[consultas_paciente['RECURRENCIA'] == 1]
    
    if len(consultas_recurrencia) > 0:
        consulta_recurrencia = consultas_recurrencia.iloc[0]
        fecha_recurrencia = consulta_recurrencia['fecha']
        recurrencia = 1
    else:
        fecha_recurrencia = None
        recurrencia = 0
    
    results.append({
        'id_paciente': paciente_id,
        'fecha_primera_consulta': fecha_primera,
        'fecha_consulta_recurrencia': fecha_recurrencia,
        'edad': edad,
        'recurrencia': recurrencia
    })

df_final = pd.DataFrame(results)

print(f"\nResultado final:")
print(f"  Total pacientes: {len(df_final)}")
print(f"  Pacientes con recurrencia: {df_final['recurrencia'].sum()}")
print(f"  Pacientes sin recurrencia: {(df_final['recurrencia'] == 0).sum()}")

print(df_final.head(10).to_string(index=False))
df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\nArchivo guardado: {OUTPUT_CSV}")


Pacientes con edad detectada: 162
Total consultas: 16603

Resultado final:
  Total pacientes: 162
  Pacientes con recurrencia: 162
  Pacientes sin recurrencia: 0
                         id_paciente fecha_primera_consulta fecha_consulta_recurrencia    edad  recurrencia
0203702C-F17D-4C01-857C-C97F46CF476F             2011/10/07                 2012/02/15 82 años            1
0401FF9E-D16D-42BA-875A-E87B9CFC08F5             2011/12/14                 2016/01/04 38 años            1
0458E951-1E16-4A82-BDE4-F399062E6D6C             2021/03/29                 2022/04/19 46 años            1
04736CA4-20C8-49B6-9641-07FFE1C0E04D             2014/08/11                 2016/02/24 43 años            1
055F0CC5-5ADC-4153-BA31-E5186DF481BE             2018/03/08                 2022/02/09 73 años            1
061BB837-DCB1-4785-991E-E7D2A3E8F527             2020/11/11                 2022/10/05 40 años            1
078C79B6-BA71-4C8F-A82D-2BFA2CE9C3E5             2013/12/11                 2017/0

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("Cargando datos...")
df = pd.read_csv(INPUT_CSV, encoding='utf-8')

# Extraer edad numerica
def extract_age(edad_str):
    if pd.isna(edad_str):
        return None
    edad_str = str(edad_str).strip().lower()
    edad_str = edad_str.replace('años', '').replace('ano', '').replace('a', '').strip()
    try:
        edad_num = int(''.join(filter(str.isdigit, edad_str)))
        return edad_num
    except:
        return None

df['edad_numerica'] = df['edad'].apply(extract_age)
df_valid = df[df['edad_numerica'].notna()].copy()

# Convertir fechas
df_valid['fecha_primera'] = pd.to_datetime(df_valid['fecha_primera_consulta'], format='%Y/%m/%d', errors='coerce')
df_valid['fecha_recurrencia'] = pd.to_datetime(df_valid['fecha_consulta_recurrencia'], format='%Y/%m/%d', errors='coerce')

# Filtrar solo pacientes con recurrencia
df_rec = df_valid[df_valid['recurrencia'] == 1].copy()
df_rec['dias_transcurridos'] = (df_rec['fecha_recurrencia'] - df_rec['fecha_primera']).dt.days
df_rec = df_rec[df_rec['dias_transcurridos'] >= 0].copy()

print(f"Pacientes con recurrencia y datos validos: {len(df_rec)}")
df_rec['rango_edad'] = pd.cut(
    df_rec['edad_numerica'], 
    bins=[0, 40, 50, 60, 70, 100], 
    labels=['<40', '40-49', '50-59', '60-69', '70+']
)

print("\nEstadisticas generales:")
print(f"  Edad media: {df_rec['edad_numerica'].mean():.1f} años")
print(f"  Dias transcurridos (media): {df_rec['dias_transcurridos'].mean():.1f}")
print(f"  Dias transcurridos (mediana): {df_rec['dias_transcurridos'].median():.1f}")

print("\nEstadisticas por rango de edad:")
print("-" * 80)

stats_rangos = df_rec.groupby('rango_edad', observed=True).agg({
    'edad_numerica': ['mean', 'count'],
    'dias_transcurridos': ['mean', 'median', 'std', 'min', 'max']
}).round(1)

print(stats_rangos)

# Guardar estadisticas
stats_export = stats_rangos.copy()
stats_export.columns = ['_'.join(col).strip() for col in stats_export.columns.values]
stats_export.to_csv(OUTPUT_STATS, encoding='utf-8')
print(f"\nEstadisticas guardadas: {OUTPUT_STATS}")

correlacion = df_rec['edad_numerica'].corr(df_rec['dias_transcurridos'])
print(f"\nCorrelacion edad vs dias transcurridos: {correlacion:.3f}")

# Gráficas
# 1. Histograma de dias transcurridos por rango de edad
fig, ax = plt.subplots(figsize=(14, 8))
for rango in df_rec['rango_edad'].cat.categories:
    data = df_rec[df_rec['rango_edad'] == rango]['dias_transcurridos']
    ax.hist(data, alpha=0.5, label=f'{rango} (n={len(data)})', bins=20)
ax.set_xlabel('Dias transcurridos', fontsize=12)
ax.set_ylabel('Frecuencia', fontsize=12)
ax.set_title('Distribucion del tiempo hasta recurrencia por rango de edad', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}histograma_por_rango.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}histograma_por_rango.png")
plt.close()

# 2. Boxplot por rango de edad
fig, ax = plt.subplots(figsize=(12, 8))
df_rec.boxplot(column='dias_transcurridos', by='rango_edad', ax=ax)
ax.set_xlabel('Rango de edad', fontsize=12)
ax.set_ylabel('Dias transcurridos', fontsize=12)
ax.set_title('Tiempo hasta recurrencia por rango de edad', fontsize=14, fontweight='bold')
plt.suptitle('')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}boxplot_por_rango.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}boxplot_por_rango.png")
plt.close()

# 3. Violin plot
fig, ax = plt.subplots(figsize=(12, 8))
sns.violinplot(data=df_rec, x='rango_edad', y='dias_transcurridos', ax=ax)
ax.set_xlabel('Rango de edad', fontsize=12)
ax.set_ylabel('Dias transcurridos', fontsize=12)
ax.set_title('Distribucion del tiempo hasta recurrencia (Violin plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}violin_plot.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}violin_plot.png")
plt.close()

# 4. Scatter plot edad vs dias
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(df_rec['edad_numerica'], df_rec['dias_transcurridos'], 
                     c=df_rec['edad_numerica'], cmap='viridis', alpha=0.6, s=100)
ax.set_xlabel('Edad (años)', fontsize=12)
ax.set_ylabel('Dias transcurridos', fontsize=12)
ax.set_title(f'Relacion edad vs tiempo hasta recurrencia (Corr: {correlacion:.3f})', 
             fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Edad')
# Añadir linea de tendencia
z = np.polyfit(df_rec['edad_numerica'], df_rec['dias_transcurridos'], 1)
p = np.poly1d(z)
ax.plot(df_rec['edad_numerica'].sort_values(), p(df_rec['edad_numerica'].sort_values()), 
        "r--", alpha=0.8, linewidth=2, label='Tendencia lineal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}scatter_edad_vs_dias.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}scatter_edad_vs_dias.png")
plt.close()

# 5. Barplot de medias por rango
fig, ax = plt.subplots(figsize=(12, 8))
medias = df_rec.groupby('rango_edad', observed=True)['dias_transcurridos'].mean()
conteos = df_rec.groupby('rango_edad', observed=True).size()
bars = ax.bar(medias.index.astype(str), medias.values, alpha=0.7, edgecolor='black')
ax.set_xlabel('Rango de edad', fontsize=12)
ax.set_ylabel('Dias transcurridos (media)', fontsize=12)
ax.set_title('Tiempo medio hasta recurrencia por rango de edad', fontsize=14, fontweight='bold')
# Añadir etiquetas con n y valor
for bar, media, n in zip(bars, medias.values, conteos.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{media:.0f} dias\nn={n}',
            ha='center', va='bottom', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}barplot_medias.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}barplot_medias.png")
plt.close()

# 6. Heatmap de estadisticas
fig, ax = plt.subplots(figsize=(10, 6))
stats_heatmap = df_rec.groupby('rango_edad', observed=True)['dias_transcurridos'].agg([
    ('Media', 'mean'),
    ('Mediana', 'median'),
    ('Desv_Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).T
sns.heatmap(stats_heatmap, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Dias'})
ax.set_title('Mapa de calor: Estadisticas por rango de edad', fontsize=14, fontweight='bold')
ax.set_xlabel('Rango de edad', fontsize=12)
ax.set_ylabel('Estadistica', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}heatmap_estadisticas.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}heatmap_estadisticas.png")
plt.close()

# 7. Distribucion acumulada
fig, ax = plt.subplots(figsize=(12, 8))
for rango in df_rec['rango_edad'].cat.categories:
    data = df_rec[df_rec['rango_edad'] == rango]['dias_transcurridos'].sort_values()
    cumulative = np.arange(1, len(data) + 1) / len(data) * 100
    ax.plot(data, cumulative, marker='o', linestyle='-', label=f'{rango} (n={len(data)})', alpha=0.7)
ax.set_xlabel('Dias transcurridos', fontsize=12)
ax.set_ylabel('Porcentaje acumulado (%)', fontsize=12)
ax.set_title('Distribucion acumulada del tiempo hasta recurrencia', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}distribucion_acumulada.png', dpi=300)
print(f"Guardado: {OUTPUT_DIR}distribucion_acumulada.png")
plt.close()

# TABLA RESUMEN DETALLADA
print("\n\nResumen detallado por rango de edad:")
print("-" * 100)

for rango in df_rec['rango_edad'].cat.categories:
    data = df_rec[df_rec['rango_edad'] == rango]['dias_transcurridos']
    if len(data) == 0:
        continue
    
    print(f"\nRango: {rango}")
    print(f"  Pacientes: {len(data)}")
    print(f"  Edad media: {df_rec[df_rec['rango_edad'] == rango]['edad_numerica'].mean():.1f} años")
    print(f"  Dias transcurridos:")
    print(f"    Media: {data.mean():.1f}")
    print(f"    Mediana: {data.median():.1f}")
    print(f"    Desv. Std: {data.std():.1f}")
    print(f"    Min: {data.min():.0f}")
    print(f"    Max: {data.max():.0f}")
    print(f"    Q1 (25%): {data.quantile(0.25):.0f}")
    print(f"    Q3 (75%): {data.quantile(0.75):.0f}")
    print(f"    Rango intercuartilico: {data.quantile(0.75) - data.quantile(0.25):.0f}")

print(f"\n\nTodas las graficas guardadas en: {OUTPUT_DIR}")


Cargando datos...
Pacientes con recurrencia y datos validos: 152

Estadisticas generales:
  Edad media: 56.1 años
  Dias transcurridos (media): 1192.4
  Dias transcurridos (mediana): 852.0

Estadisticas por rango de edad:
--------------------------------------------------------------------------------
           edad_numerica       dias_transcurridos                          
                    mean count               mean median     std  min   max
rango_edad                                                                 
<40                 36.3    23              935.1  693.0   735.4   40  3398
40-49               45.6    47             1287.6  883.0  1180.8   71  4763
50-59               55.0    26             1211.4  855.0   900.6  119  3826
60-69               66.7    23             1028.1  903.0   655.4   14  2418
70+                 78.3    33             1335.5  786.0  1354.6   70  5382

Estadisticas guardadas: recurrence_stats_by_age.csv

Correlacion edad vs dias transcurri